In [1]:
import pandas as pd
import numpy as np
import torch

print("PyTorch version:", torch.__version__)

df = pd.read_csv("../data/Base.csv")

print("Dataset shape:", df.shape)
print(df.head())

PyTorch version: 2.14.0+cpu
Dataset shape: (1000000, 32)
   fraud_bool  income  name_email_similarity  prev_address_months_count  \
0           0     0.3               0.986506                         -1   
1           0     0.8               0.617426                         -1   
2           0     0.8               0.996707                          9   
3           0     0.6               0.475100                         11   
4           0     0.9               0.842307                         -1   

   current_address_months_count  customer_age  days_since_request  \
0                            25            40            0.006735   
1                            89            20            0.010095   
2                            14            40            0.012316   
3                            14            30            0.006991   
4                            29            40            5.742626   

   intended_balcon_amount payment_type  zip_count_4w  ...  has_other_cards  \

In [2]:
print("Columns:")
for i, col in enumerate(df.columns):
    print(i, col)

Columns:
0 fraud_bool
1 income
2 name_email_similarity
3 prev_address_months_count
4 current_address_months_count
5 customer_age
6 days_since_request
7 intended_balcon_amount
8 payment_type
9 zip_count_4w
10 velocity_6h
11 velocity_24h
12 velocity_4w
13 bank_branch_count_8w
14 date_of_birth_distinct_emails_4w
15 employment_status
16 credit_risk_score
17 email_is_free
18 housing_status
19 phone_home_valid
20 phone_mobile_valid
21 bank_months_count
22 has_other_cards
23 proposed_credit_limit
24 foreign_request
25 source
26 session_length_in_minutes
27 device_os
28 keep_alive_session
29 device_distinct_emails_8w
30 device_fraud_count
31 month


In [3]:
print("\nData types:")
print(df.dtypes)


Data types:
fraud_bool                            int64
income                              float64
name_email_similarity               float64
prev_address_months_count             int64
current_address_months_count          int64
customer_age                          int64
days_since_request                  float64
intended_balcon_amount              float64
payment_type                            str
zip_count_4w                          int64
velocity_6h                         float64
velocity_24h                        float64
velocity_4w                         float64
bank_branch_count_8w                  int64
date_of_birth_distinct_emails_4w      int64
employment_status                       str
credit_risk_score                     int64
email_is_free                         int64
housing_status                          str
phone_home_valid                      int64
phone_mobile_valid                    int64
bank_months_count                     int64
has_other_cards    

In [4]:
print("\nUnique values:")
print(df.nunique().sort_values())


Unique values:
device_fraud_count                       1
fraud_bool                               2
keep_alive_session                       2
foreign_request                          2
phone_home_valid                         2
has_other_cards                          2
email_is_free                            2
source                                   2
phone_mobile_valid                       2
device_distinct_emails_8w                4
device_os                                5
payment_type                             5
employment_status                        7
housing_status                           7
month                                    8
income                                   9
customer_age                             9
proposed_credit_limit                   12
bank_months_count                       33
date_of_birth_distinct_emails_4w        40
prev_address_months_count              374
current_address_months_count           423
credit_risk_score                     

In [5]:
# Check categorical columns and their cardinality

categorical_cols = df.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical columns:")
for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values")

Categorical columns:
payment_type: 5 unique values
employment_status: 7 unique values
housing_status: 7 unique values
source: 2 unique values
device_os: 5 unique values


C:\Users\dell\AppData\Local\Temp\ipykernel_5148\292745713.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(


In [6]:
# Show all columns with relatively low cardinality

print("Low-cardinality features:")

for col in df.columns:
    unique_count = df[col].nunique()

    if unique_count <= 100:
        print(f"{col}: {unique_count} unique values")

Low-cardinality features:
fraud_bool: 2 unique values
income: 9 unique values
customer_age: 9 unique values
payment_type: 5 unique values
date_of_birth_distinct_emails_4w: 40 unique values
employment_status: 7 unique values
email_is_free: 2 unique values
housing_status: 7 unique values
phone_home_valid: 2 unique values
phone_mobile_valid: 2 unique values
bank_months_count: 33 unique values
has_other_cards: 2 unique values
proposed_credit_limit: 12 unique values
foreign_request: 2 unique values
source: 2 unique values
device_os: 5 unique values
keep_alive_session: 2 unique values
device_distinct_emails_8w: 4 unique values
device_fraud_count: 1 unique values
month: 8 unique values


In [7]:
# Display the categorical values

for col in categorical_cols:
    print(f"\n{col}:")
    print(df[col].value_counts().head(20))


payment_type:
payment_type
AB    370554
AA    258249
AC    252071
AD    118837
AE       289
Name: count, dtype: int64

employment_status:
employment_status
CA    730252
CB    138288
CF     44034
CC     37758
CD     26522
CE     22693
CG       453
Name: count, dtype: int64

housing_status:
housing_status
BC    372143
BB    260965
BA    169675
BE    169135
BD     26161
BF      1669
BG       252
Name: count, dtype: int64

source:
source
INTERNET    992952
TELEAPP       7048
Name: count, dtype: int64

device_os:
device_os
other        342728
linux        332712
windows      263506
macintosh     53826
x11            7228
Name: count, dtype: int64


In [8]:
# Temporal split — same as XGBoost experiments

train_df = df[df["month"] < 5].copy()

val_df = df[
    (df["month"] >= 5) &
    (df["month"] < 6)
].copy()

test_df = df[df["month"] >= 6].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (675666, 32)
Validation: (119323, 32)
Test: (205011, 32)


In [9]:
# Separate features and target

X_train = train_df.drop(columns=["fraud_bool"])
y_train = train_df["fraud_bool"]

X_val = val_df.drop(columns=["fraud_bool"])
y_val = val_df["fraud_bool"]

X_test = test_df.drop(columns=["fraud_bool"])
y_test = test_df["fraud_bool"]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (675666, 31)
X_val: (119323, 31)
X_test: (205011, 31)


In [10]:
# Identify categorical and numerical columns

categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_cols = X_train.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Number of categorical columns:", len(categorical_cols))

print("\nNumerical columns:", numerical_cols)
print("Number of numerical columns:", len(numerical_cols))

Categorical columns: ['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']
Number of categorical columns: 5

Numerical columns: ['income', 'name_email_similarity', 'prev_address_months_count', 'current_address_months_count', 'customer_age', 'days_since_request', 'intended_balcon_amount', 'zip_count_4w', 'velocity_6h', 'velocity_24h', 'velocity_4w', 'bank_branch_count_8w', 'date_of_birth_distinct_emails_4w', 'credit_risk_score', 'email_is_free', 'phone_home_valid', 'phone_mobile_valid', 'bank_months_count', 'has_other_cards', 'proposed_credit_limit', 'foreign_request', 'session_length_in_minutes', 'keep_alive_session', 'device_distinct_emails_8w', 'device_fraud_count', 'month']
Number of numerical columns: 26


C:\Users\dell\AppData\Local\Temp\ipykernel_5148\2433422085.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(


In [11]:
# Convert categorical columns to category codes
# using the training data as the reference

for col in categorical_cols:
    categories = X_train[col].astype("category").cat.categories

    X_train[col] = pd.Categorical(
        X_train[col],
        categories=categories
    ).codes

    X_val[col] = pd.Categorical(
        X_val[col],
        categories=categories
    ).codes

    X_test[col] = pd.Categorical(
        X_test[col],
        categories=categories
    ).codes

print("Categorical features encoded.")

Categorical features encoded.


In [12]:
print(X_train.dtypes)

income                              float64
name_email_similarity               float64
prev_address_months_count             int64
current_address_months_count          int64
customer_age                          int64
days_since_request                  float64
intended_balcon_amount              float64
payment_type                           int8
zip_count_4w                          int64
velocity_6h                         float64
velocity_24h                        float64
velocity_4w                         float64
bank_branch_count_8w                  int64
date_of_birth_distinct_emails_4w      int64
employment_status                      int8
credit_risk_score                     int64
email_is_free                         int64
housing_status                         int8
phone_home_valid                      int64
phone_mobile_valid                    int64
bank_months_count                     int64
has_other_cards                       int64
proposed_credit_limit           

In [13]:
# Remove month from graph features
# It was used only for temporal splitting

X_train = X_train.drop(columns=["month"])
X_val = X_val.drop(columns=["month"])
X_test = X_test.drop(columns=["month"])

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (675666, 30)
X_val shape: (119323, 30)
X_test shape: (205011, 30)


In [14]:
from sklearn.preprocessing import StandardScaler

# Convert to float32 for PyTorch
X_train_np = X_train.astype(np.float32).values
X_val_np = X_val.astype(np.float32).values
X_test_np = X_test.astype(np.float32).values

# Fit scaler only on training data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_np)
X_val_scaled = scaler.transform(X_val_np)
X_test_scaled = scaler.transform(X_test_np)

print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

print("\nMean of first 5 scaled features:")
print(X_train_scaled[:, :5].mean(axis=0))

print("\nStd of first 5 scaled features:")
print(X_train_scaled[:, :5].std(axis=0))

Train: (675666, 30)
Validation: (119323, 30)
Test: (205011, 30)

Mean of first 5 scaled features:
[ 3.1797331e-08 -3.3242664e-08  2.3125331e-08 -3.1797331e-08
 -1.8789331e-08]

Std of first 5 scaled features:
[1. 1. 1. 1. 1.]


In [15]:
from sklearn.neighbors import NearestNeighbors

# Number of graph neighbors
k = 5

knn = NearestNeighbors(
    n_neighbors=k + 1,
    metric="euclidean",
    algorithm="auto",
    n_jobs=-1
)

knn.fit(X_train_scaled)

print("k-NN model fitted.")

k-NN model fitted.


In [16]:
# Install/import a faster approximate nearest-neighbor library

import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "pynndescent",
    "-q"
])

from pynndescent import NNDescent

print("PyNNDescent ready!")

PyNNDescent ready!


In [17]:
# Build an approximate k-NN graph

k = 5

nn_descent = NNDescent(
    X_train_scaled,
    n_neighbors=k + 1,
    metric="euclidean",
    random_state=42,
    n_jobs=-1
)

indices, distances = nn_descent.neighbor_graph

print("Graph construction completed!")
print("Indices shape:", indices.shape)
print("Distances shape:", distances.shape)

Graph construction completed!
Indices shape: (675666, 6)
Distances shape: (675666, 6)


In [18]:
# Remove self-neighbor and create graph edges

n_nodes = len(X_train_scaled)

# Remove the first neighbor (self)
neighbor_indices = indices[:, 1:]
neighbor_distances = distances[:, 1:]

print("Neighbor indices shape:", neighbor_indices.shape)
print("Neighbor distances shape:", neighbor_distances.shape)

# Create source nodes
source_nodes = np.repeat(
    np.arange(n_nodes),
    k
)

# Flatten destination nodes
destination_nodes = neighbor_indices.reshape(-1)

# Create edge index
edge_index = np.vstack([
    source_nodes,
    destination_nodes
])

print("\nGraph created!")
print("Number of nodes:", n_nodes)
print("Number of edges:", edge_index.shape[1])
print("Edge index shape:", edge_index.shape)

Neighbor indices shape: (675666, 5)
Neighbor distances shape: (675666, 5)

Graph created!
Number of nodes: 675666
Number of edges: 3378330
Edge index shape: (2, 3378330)


In [19]:
# Check a few graph edges

print("First 10 source nodes:")
print(edge_index[0, :10])

print("\nFirst 10 destination nodes:")
print(edge_index[1, :10])

print("\nTotal edges:", edge_index.shape[1])

First 10 source nodes:
[0 0 0 0 0 1 1 1 1 1]

First 10 destination nodes:
[  4871  53324 109543 108988 102937  34483  51557  19587  83066  80333]

Total edges: 3378330


In [20]:
# Convert graph data to PyTorch tensors

import torch

# Node features
x = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

# Labels
y = torch.tensor(
    y_train.values,
    dtype=torch.long
)

# Graph edges
edge_index_tensor = torch.tensor(
    edge_index,
    dtype=torch.long
)

print("Node features:", x.shape)
print("Labels:", y.shape)
print("Edge index:", edge_index_tensor.shape)

print("\nFraud nodes:", (y == 1).sum().item())
print("Non-fraud nodes:", (y == 0).sum().item())

Node features: torch.Size([675666, 30])
Labels: torch.Size([675666])
Edge index: torch.Size([2, 3378330])

Fraud nodes: 6740
Non-fraud nodes: 668926


In [21]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cpu


In [22]:
# Install PyTorch Geometric

import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "torch-geometric",
    "-q"
])

print("PyTorch Geometric installed!")

PyTorch Geometric installed!


In [23]:
import torch_geometric

print("PyTorch Geometric version:", torch_geometric.__version__)

d:\ML project\FairFraud\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


PyTorch Geometric version: 2.8.0.post1


In [24]:
from torch_geometric.data import Data

# Create PyTorch Geometric graph

graph_data = Data(
    x=x,
    edge_index=edge_index_tensor,
    y=y
)

print(graph_data)
print("\nNodes:", graph_data.num_nodes)
print("Edges:", graph_data.num_edges)
print("Features per node:", graph_data.num_node_features)

Data(x=[675666, 30], edge_index=[2, 3378330], y=[675666])

Nodes: 675666
Edges: 3378330
Features per node: 30


In [25]:
# Create train/validation masks

num_nodes = graph_data.num_nodes

train_mask = torch.zeros(num_nodes, dtype=torch.bool)

train_mask[:] = True

graph_data.train_mask = train_mask

print("Training nodes:", graph_data.train_mask.sum().item())

Training nodes: 675666


In [26]:
from torch_geometric.nn import SAGEConv
import torch.nn.functional as F


class GraphSAGE(torch.nn.Module):

    def __init__(self, input_dim, hidden_dim=64, output_dim=2):
        super().__init__()

        self.conv1 = SAGEConv(input_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)

        self.classifier = torch.nn.Linear(
            hidden_dim,
            output_dim
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        x = self.classifier(x)

        return x

In [27]:
# Create GraphSAGE model

model = GraphSAGE(
    input_dim=graph_data.num_node_features,
    hidden_dim=64,
    output_dim=2
)

print(model)

GraphSAGE(
  (conv1): SAGEConv(30, 64, aggr=mean)
  (conv2): SAGEConv(64, 64, aggr=mean)
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)


In [28]:
# Count trainable parameters

total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Trainable parameters:", total_params)

Trainable parameters: 12290


In [29]:
from torch_geometric.loader import NeighborLoader

# Neighbor sampling for CPU training

train_loader = NeighborLoader(
    graph_data,
    num_neighbors=[5, 5],
    batch_size=1024,
    input_nodes=graph_data.train_mask,
    shuffle=True,
    num_workers=0
)

print("NeighborLoader created!")
print("Batches per epoch:", len(train_loader))

NeighborLoader created!
Batches per epoch: 660


In [30]:
# Inspect one sampled batch

batch = next(iter(train_loader))

print("Batch nodes:", batch.num_nodes)
print("Batch edges:", batch.edge_index.shape[1])
print("Seed nodes:", batch.batch_size)
print("Features:", batch.x.shape)
print("Labels:", batch.y.shape)

Batch nodes: 13409
Batch edges: 15189
Seed nodes: 1024
Features: torch.Size([13409, 30])
Labels: torch.Size([13409])


In [31]:
print("PyTorch:", torch.__version__)
print("PyG:", torch_geometric.__version__)

PyTorch: 2.14.0+cpu
PyG: 2.8.0.post1


In [32]:
import torch
import torch.nn.functional as F

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

criterion = torch.nn.CrossEntropyLoss()

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        batch = batch.to(device)

        out = model(batch.x, batch.edge_index)

        # Only calculate loss on seed nodes
        loss = criterion(
            out[:batch.batch_size],
            batch.y[:batch.batch_size]
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

Epoch 1/5, Loss: 0.0539
Epoch 2/5, Loss: 0.0455
Epoch 3/5, Loss: 0.0450
Epoch 4/5, Loss: 0.0448
Epoch 5/5, Loss: 0.0447


In [33]:
from sklearn.neighbors import NearestNeighbors

# Use training nodes as the reference graph
k_eval = 5

nn_eval = NearestNeighbors(
    n_neighbors=k_eval,
    metric="euclidean",
    n_jobs=-1
)

nn_eval.fit(X_train_scaled)

print("Reference training graph ready.")

Reference training graph ready.


In [ ]:
# Find the 5 nearest training nodes for every validation node

val_distances, val_neighbors = nn_eval.kneighbors(
    X_val_scaled,
    return_distance=True
)

print("Validation neighbors shape:", val_neighbors.shape)
print("Validation distances shape:", val_distances.shape)